# Trash Tracker Training

Trains a YOLO26 detector for the six waste classes and exports TFLite weights for the Flutter app.

Requires a `ROBOFLOW_API_KEY` entry in `notebooks/.env` (see `.env.example`). Dataset must include unlabeled negative/background images to keep false positives down.

In [1]:
#!pip install -r requirements.txt
#%pip install roboflow

In [2]:
import os
import shutil
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv
import numpy as np
import yaml
from PIL import Image
from roboflow import Roboflow
from ultralytics import YOLO

try:
    import tensorflow as tf
except ImportError:
    tf = None

## 1. Configuration

In [3]:
# --- Roboflow (change these when you create a new dataset version) ---
ROBOFLOW_WORKSPACE = "damians-workspace-enpcl"
ROBOFLOW_PROJECT = "trash_tracker"  # or your new project slug
ROBOFLOW_VERSION = 5  # bump when you upload new data

# --- Training run name (weights saved under ml_output/runs/detect/<RUN_NAME>/) ---
RUN_NAME = "trash_tracker_v5b"

# --- Paths ---
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

# Keep dataset downloads, training runs, and test images out of the repo root.
# Roboflow's download() and YOLO's train()/val() both write relative to cwd
# when no explicit path is given, so moving cwd here once keeps everything
# under one folder no matter where this notebook is launched from.
ML_OUTPUT_DIR = REPO_ROOT / "ml_output"
ML_OUTPUT_DIR.mkdir(exist_ok=True)
os.chdir(ML_OUTPUT_DIR)

TEST_IMAGES_DIR = ML_OUTPUT_DIR / "test_images"
TEST_IMAGES_DIR.mkdir(exist_ok=True)

# Must match assets/labels.txt and app class IDs (0-5)
APP_CLASS_ORDER = [
    "BIODEGRADABLE",
    "CARDBOARD",
    "GLASS",
    "METAL",
    "PAPER",
    "PLASTIC",
]

# --- Model / training ---
BASE_WEIGHTS = "yolo26s.pt"  # use a .pt path to fine-tune instead
IMG_SIZE = 800  # must match Flutter app letterbox input
EPOCHS = 150
BATCH_SIZE = 16  # lower to 8 if GPU runs out of memory
DEVICE = 0  # GPU index; use "cpu" if no CUDA
WORKERS = 0  # Windows: keep 0; Linux can use 4-8

# Loads notebooks/.env (create it from .env.example, next to this notebook).
# Keeps the key out of your shell history and out of this notebook entirely.
load_dotenv(NOTEBOOK_DIR / ".env")
API_KEY = os.environ.get("ROBOFLOW_API_KEY")
if not API_KEY:
    raise EnvironmentError(
        "ROBOFLOW_API_KEY not found. Copy notebooks/.env.example to notebooks/.env "
        "and fill in your key (do not hard-code it here)."
    )

print(f"Repo root: {REPO_ROOT}")
print(f"ML output dir: {ML_OUTPUT_DIR}")
print(f"Run name: {RUN_NAME}")
print(f"Roboflow: {ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT} v{ROBOFLOW_VERSION}")

Repo root: C:\Users\damia\Documents\repositories\trash_tracker
ML output dir: C:\Users\damia\Documents\repositories\trash_tracker\ml_output
Run name: trash_tracker_v5b
Roboflow: damians-workspace-enpcl/trash_tracker v5


## 2. Download dataset

Exports the Roboflow project in YOLO26 format and loads `data.yaml`.

In [4]:
rf = Roboflow(api_key=API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download("yolo26")

DATA_YAML = Path(dataset.location) / "data.yaml"
print(f"Dataset root: {dataset.location}")
print(f"data.yaml: {DATA_YAML}")

with open(DATA_YAML, encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

print("\ndata.yaml contents:")
print(yaml.dump(data_cfg, default_flow_style=False))

loading Roboflow workspace...
loading Roboflow project...
Dataset root: C:\Users\damia\Documents\repositories\trash_tracker\ml_output\trash_tracker-5
data.yaml: C:\Users\damia\Documents\repositories\trash_tracker\ml_output\trash_tracker-5\data.yaml

data.yaml contents:
names:
- BIODEGRADABLE
- CARDBOARD
- GLASS
- METAL
- PAPER
- PLASTIC
nc: 6
roboflow:
  license: CC BY 4.0
  project: trash_tracker
  url: https://universe.roboflow.com/damians-workspace-enpcl/trash_tracker/dataset/5
  version: 5
  workspace: damians-workspace-enpcl
test: ../test/images
train: ../train/images
val: ../valid/images



## 3. Dataset audit

Checks class order and negative/instance counts before training.

In [5]:
def resolve_split_dir(data_yaml_path: Path, split_key: str):
    # Roboflow's data.yaml often points to "../train/images", which assumes a
    # nested folder layout that the roboflow package's own download() doesn't
    # actually produce. Try the yaml-specified path first, then fall back to
    # Roboflow's real on-disk layout: <root>/<split>/images.
    with open(data_yaml_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    root = data_yaml_path.parent
    candidates = []
    raw = cfg.get(split_key)
    if raw:
        p = Path(raw)
        candidates.append(p if p.is_absolute() else (root / p))
    folder_name = {"val": "valid"}.get(split_key, split_key)
    candidates.append(root / folder_name / "images")
    for c in candidates:
        if c.exists():
            return c
    return candidates[0] if candidates else None


def audit_yolo_split(images_dir: Path, names: dict[int, str]) -> dict:
    if images_dir is None or not images_dir.exists():
        return {"images": 0, "negatives": 0, "instances": Counter(), "images_with_boxes": 0}

    labels_dir = images_dir.parent / "labels"
    if not labels_dir.exists():
        labels_dir = images_dir.with_name("labels")

    instance_counts = Counter()
    negatives = 0
    with_boxes = 0
    image_files = [
        p for p in images_dir.glob("*.*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    ]

    for img_path in image_files:
        label_path = labels_dir / f"{img_path.stem}.txt"
        if not label_path.exists() or label_path.stat().st_size == 0:
            negatives += 1
            continue
        lines = [
            ln.strip()
            for ln in label_path.read_text(encoding="utf-8").splitlines()
            if ln.strip()
        ]
        if not lines:
            negatives += 1
            continue
        with_boxes += 1
        for line in lines:
            class_id = int(line.split()[0])
            label = names.get(class_id, f"unknown_{class_id}")
            instance_counts[label] += 1

    return {
        "images": len(image_files),
        "negatives": negatives,
        "images_with_boxes": with_boxes,
        "instances": instance_counts,
    }


with open(DATA_YAML, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

names = cfg.get("names", {})
if isinstance(names, list):
    names = {i: n for i, n in enumerate(names)}
else:
    names = {int(k): v for k, v in names.items()}

yaml_order = [names[i] for i in sorted(names)]
print("Classes in data.yaml (by ID):")
for i, name in enumerate(yaml_order):
    print(f"  {i}: {name}")

if yaml_order != APP_CLASS_ORDER:
    print("\n⚠️  WARNING: class order does not match assets/labels.txt!")
    print(f"  Expected: {APP_CLASS_ORDER}")
    print(f"  Got:      {yaml_order}")
    print("  Fix class order in Roboflow before training, or update assets/labels.txt to match.")
else:
    print("\n✓ Class order matches the Flutter app.")

for split in ("train", "val", "valid", "test"):
    split_dir = resolve_split_dir(DATA_YAML, split)
    if split_dir is None:
        continue
    stats = audit_yolo_split(split_dir, names)
    print(f"\n--- {split.upper()} ---")
    print(f"  Images: {stats['images']}")
    print(f"  With boxes: {stats['images_with_boxes']}")
    print(f"  Negatives (no boxes): {stats['negatives']}")
    if stats["negatives"] == 0:
        print("  ⚠️  No negative images — expect false positives on desks/monitors.")
    print("  Instances per class:")
    for label, count in stats["instances"].most_common():
        print(f"    {label:15} {count}")

Classes in data.yaml (by ID):
  0: BIODEGRADABLE
  1: CARDBOARD
  2: GLASS
  3: METAL
  4: PAPER
  5: PLASTIC

✓ Class order matches the Flutter app.

--- TRAIN ---
  Images: 3042
  With boxes: 2958
  Negatives (no boxes): 84
  Instances per class:
    PLASTIC         1250
    BIODEGRADABLE   1206
    METAL           795
    CARDBOARD       654
    PAPER           621
    GLASS           609

--- VAL ---
  Images: 448
  With boxes: 440
  Negatives (no boxes): 8
  Instances per class:
    BIODEGRADABLE   201
    CARDBOARD       155
    GLASS           145
    PLASTIC         134
    METAL           87
    PAPER           60

--- VALID ---
  Images: 448
  With boxes: 440
  Negatives (no boxes): 8
  Instances per class:
    BIODEGRADABLE   201
    CARDBOARD       155
    GLASS           145
    PLASTIC         134
    METAL           87
    PAPER           60

--- TEST ---
  Images: 370
  With boxes: 366
  Negatives (no boxes): 4
  Instances per class:
    PLASTIC         195
    BIODEGRA

## 4. Train

Trains from `BASE_WEIGHTS`. `workers=0` avoids Windows dataloader issues.

In [6]:
model = YOLO(BASE_WEIGHTS)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    name=RUN_NAME,
    optimizer="MuSGD",
    lr0=0.001,
    cos_lr=True,
    # Augmentations
    mosaic=1.0,
    mixup=0.1,
    degrees=15.0,
    flipud=0.1,  # low — trash usually sits upright; was 0.5 in Phase 4
    fliplr=0.5,
    patience=30,
    save_period=10,
)

BEST_WEIGHTS = Path(results.save_dir) / "weights" / "best.pt"
print(f"\nBest weights: {BEST_WEIGHTS}")

WARNING Corrupt cache yolo26s.pt, re-downloading yolo26s.pt...
Ultralytics 8.4.115  Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\damia\Documents\repositories\trash_tracker\ml_output\trash_tracker-5\data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4

## 5. Validate on the Roboflow val split

In [7]:
model = YOLO(str(BEST_WEIGHTS))
metrics = model.val(data=str(DATA_YAML), imgsz=IMG_SIZE, name=f"{RUN_NAME}_val", exist_ok=True)

print("--- Per-class mAP50 ---")
for name, mAP in zip(metrics.names.values(), metrics.box.maps):
    print(f"{name:15}: {mAP:.4f}")
print(f"\nOverall mAP50: {metrics.box.map50:.4f}")
print(f"Overall mAP50-95: {metrics.box.map:.4f}")

Ultralytics 8.4.115  Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
YOLO26n summary (fused): 122 layers, 2,376,006 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 181.6115.2 MB/s, size: 10.9 KB)
val: Scanning C:\Users\damia\Documents\repositories\trash_tracker\trash_tracker-5\valid\labels.cache... 448 images, 8 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 448/448  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 28/28 11.2it/s 2.5s0.1s
                   all        448        782      0.745      0.626      0.694      0.555
         BIODEGRADABLE         91        201      0.766      0.662      0.744       0.58
             CARDBOARD         96        155      0.838      0.523      0.637        0.5
                 GLASS        113        145      0.796      0.809      0.863      0.686
                 METAL         73         87      0.694      0.667      0.705

## 6. Test on real images

Drop images into `test_images/` and check that negatives get zero boxes and real items are classed correctly.

In [8]:
model = YOLO(str(BEST_WEIGHTS))
test_images = sorted(
    p for p in TEST_IMAGES_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
)

if not test_images:
    print(f"No images in {TEST_IMAGES_DIR}. Add desk/trash test photos and re-run.")
else:
    predict_dir = Path(results.save_dir) / "phone_tests"
    for img_path in test_images:
        print(f"\n=== {img_path.name} ===")
        preds = model.predict(
            source=str(img_path),
            imgsz=IMG_SIZE,
            conf=0.25,
            save=True,
            project=str(predict_dir),
            name=img_path.stem,
            exist_ok=True,
        )
        r = preds[0]
        print(f"Boxes: {len(r.boxes)}")
        for i, box in enumerate(r.boxes):
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            name = r.names[cls_id]
            print(f"  [{i}] {name} {conf:.2%}")
    print(f"\nSaved annotated images under: {predict_dir}")

No images in C:\Users\damia\Documents\repositories\trash_tracker\test_images. Add desk/trash test photos and re-run.


## 7. Export TFLite

App contract: input `[1, 800, 800, 3]` float RGB 0–1; output `[1, 300, 6]` rows of `x1, y1, x2, y2, confidence, class_id` (normalized xyxy).

In [9]:
model = YOLO(str(BEST_WEIGHTS))
export_path = model.export(format="tflite", imgsz=IMG_SIZE, int8=False)
export_path = Path(export_path)
print(f"Exported: {export_path}")

app_asset = REPO_ROOT / "assets" / "best_float32.tflite"
if app_asset.parent.exists():
    shutil.copy2(export_path, app_asset)
    print(f"Copied to app: {app_asset}")
else:
    print(f"Copy manually to: {app_asset}")

WARNING 'int8' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert/
Ultralytics 8.4.115  Python-3.10.20 torch-2.11.0+cu128 CPU (AMD Ryzen 7 7800X3D 8-Core Processor)
YOLO26n summary (fused): 122 layers, 2,376,006 parameters, 0 gradients, 8.4 GFLOPs

PyTorch: starting from 'C:\Users\damia\Documents\repositories\trash_tracker\runs\detect\trash_tracker_v5\weights\best.pt' with input shape (1, 3, 800, 800) BCHW and output shape(s) (1, 300, 6) (5.2 MB)
ERROR LiteRT: export failure 0.0s: LiteRT export only supported on Linux x86 and macOS
Export to tflite in the cloud with Ultralytics Platform: https://platform.ultralytics.com


AssertionError: LiteRT export only supported on Linux x86 and macOS

## 8. Sanity-check TFLite (optional)

Runs one test image through the exported TFLite model directly.

In [ ]:
if tf is None:
    print("Install tensorflow to run this cell: pip install tensorflow")
elif not test_images:
    print("Add a test image to test_images/ first.")
else:
    tflite_path = export_path if export_path.suffix == ".tflite" else export_path.with_suffix(".tflite")
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    in_det = interpreter.get_input_details()[0]
    out_det = interpreter.get_output_details()[0]
    print("TFLite input shape:", in_det["shape"])
    print("TFLite output shape:", out_det["shape"])

    img = Image.open(test_images[0]).convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = arr.reshape(1, IMG_SIZE, IMG_SIZE, 3)
    interpreter.set_tensor(in_det["index"], arr)
    interpreter.invoke()
    out = interpreter.get_tensor(out_det["index"])[0]
    good = out[out[:, 4] >= 0.25]
    print(f"Rows above conf 0.25: {len(good)}")
    if len(good):
        row = good[0]
        print(f"First row: x1={row[0]:.3f} y1={row[1]:.3f} x2={row[2]:.3f} y2={row[3]:.3f} conf={row[4]:.3f} cls={int(row[5])}")

## 9. Deploy

Confirm `assets/best_float32.tflite` and `assets/labels.txt` are updated, then `flutter run` and hot-restart (not hot reload) to pick up the new model.